In [55]:
import jax
import jax.numpy as jnp
import itertools

def vfor(**axes):
    """
    Evaluates a function over a grid of spatial axes using nested jax.vmap.
    Usage:
        @vfor(i=5, j=10)
        def C(i, j): ...
    """
    def decorator(f):
        axis_names = list(axes.keys())
        axis_sizes = list(axes.values())

        mapped_fn = f

        # Build the nested vmap from the inside out (right to left)
        # For @vfor(i=5, j=10), this maps `j` first, then maps `i`.
        for idx in reversed(range(len(axis_names))):
            in_axes = [None] * len(axis_names)
            in_axes[idx] = 0  # Map over the current axis, broadcast the rest

            mapped_fn = jax.vmap(mapped_fn, in_axes=tuple(in_axes))

        # Generate the index arrays: e.g., jnp.arange(5) and jnp.arange(10)
        index_arrays = [jnp.arange(size) for size in axis_sizes]

        # Execute the JAX graph and return the evaluated tensor
        return mapped_fn(*index_arrays)

    return decorator



In [ ]:
A = jnp.array([1, 2, 3, 4, 5])
B = jnp.arange(10)

# Approach 1: The Decorator
@vfor(i=5, j=10)
def C(i, j):
    return A[i] * B[j]

print(C.shape) 
# Output: (5, 10)
print(C)

(5, 10)
[[ 0  1  2  3  4  5  6  7  8  9]
 [ 0  2  4  6  8 10 12 14 16 18]
 [ 0  3  6  9 12 15 18 21 24 27]
 [ 0  4  8 12 16 20 24 28 32 36]
 [ 0  5 10 15 20 25 30 35 40 45]]


In [9]:
import jax
import jax.numpy as jnp

class scan:
    """Marker class to define a temporal/scanned axis."""
    def __init__(self, size):
        self.size = size

class SelfProxy:
    """Intercepts self[...] calls and returns the current JAX carry."""
    def __init__(self, carry):
        self.carry = carry

    def __getitem__(self, indices):
        # With 1 scan axis, we ignore the tuple of indices 
        # and simply return the strictly carried state.
        return self.carry

def vfor(**axes):
    """
    Evaluates a function over a grid of spatial and scanned axes.
    Usage:
        @vfor(i=5, j=10, k=scan(20))
        def C(i, j, k, self): ...
    """
    def decorator(f):
        axis_names = list(axes.keys())

        # 1. Categorize the axes
        scan_axes = {k: v for k, v in axes.items() if isinstance(v, scan)}
        spatial_axes = {k: v for k, v in axes.items() if isinstance(v, int)}

        if len(scan_axes) > 1:
            raise NotImplementedError("This prototype supports exactly 1 scan axis.")

        scan_name = next(iter(scan_axes.keys())) if scan_axes else None
        scan_size = scan_axes[scan_name].size if scan_name else None

        def execute():
            # Pure spatial fallback
            if not scan_name:
                mapped_fn = f
                for idx in reversed(range(len(axis_names))):
                    in_axes = [None] * len(axis_names)
                    in_axes[idx] = 0
                    mapped_fn = jax.vmap(mapped_fn, in_axes=tuple(in_axes))
                index_arrays = [jnp.arange(size) for size in axes.values()]
                return mapped_fn(*index_arrays)

            # --- THE SCAN COMPILER ---

            # Core function for a *single* spatial location
            def core_fn(*spatial_args):
                # Map spatial arguments back to their names
                current_args = {}
                spatial_idx = 0
                for name in axis_names:
                    if name != scan_name:
                        current_args[name] = spatial_args[spatial_idx]
                        spatial_idx += 1

                # Trace 1: The Initialization (k = -1)
                # We inject `None` to extract the initial carry state.
                current_args[scan_name] = None
                args_list = [current_args[name] for name in axis_names]
                init_carry = f(*args_list, SelfProxy(None)) 

                # Trace 2: The Main Body (k = 0 to N-1)
                def step_fn(carry, k_val):
                    current_args[scan_name] = k_val
                    args_list = [current_args[name] for name in axis_names]
                    next_val = f(*args_list, SelfProxy(carry))
                    return next_val, next_val

                # Scan natively runs exactly N times, generating the perfect grid shape
                k_indices = jnp.arange(scan_size)
                _, full_grid = jax.lax.scan(step_fn, init_carry, k_indices)

                return full_grid

            # 3. Apply the Spatial Vmaps
            spatial_sizes = list(spatial_axes.values())
            mapped_fn = core_fn

            for idx in reversed(range(len(spatial_sizes))):
                in_axes = [None] * len(spatial_sizes)
                in_axes[idx] = 0
                mapped_fn = jax.vmap(mapped_fn, in_axes=tuple(in_axes))

            spatial_index_arrays = [jnp.arange(size) for size in spatial_sizes]
            result = mapped_fn(*spatial_index_arrays)

            # 4. Transpose output to match user's signature
            current_scan_pos = len(axis_names) - 1
            target_scan_pos = axis_names.index(scan_name)

            if current_scan_pos != target_scan_pos:
                result = jnp.moveaxis(result, current_scan_pos, target_scan_pos)

            return result

        return execute()
    return decorator


In [10]:
A = jnp.array([10, 20, 30])  

@vfor(i=3, k=scan(5))
def C(i, k, self):
    # What is the state at k = -1?
    if k is None:
        return 0.0

    # Standard recursive math for k = 0, 1, 2, 3, 4
    else:
        # At k=0, self[i, k-1] beautifully pulls the 0.0 we just defined!
        return A[i] * k + self[i, k-1]

In [11]:
C

Array([[  0.,  10.,  30.,  60., 100.],
       [  0.,  20.,  60., 120., 200.],
       [  0.,  30.,  90., 180., 300.]], dtype=float32, weak_type=True)

In [37]:
import jax
import jax.numpy as jnp

class scan:
    """Marker class to define a temporal/scanned axis."""
    def __init__(self, size):
        self.size = size

class IndexProxy:
    """
    Wraps dynamic JAX loop variables. 
    Intercepts `- 1` for self indexing, but forwards all other math to JAX.
    """
    def __init__(self, name, val, offset=0):
        self.name = name
        self.val = val
        self.offset = offset

    def __sub__(self, other):
        if isinstance(other, int):
            return IndexProxy(self.name, self.val - other, self.offset - other)
        return self.val - other

    # Forward standard mathematical operations to the underlying JAX tracer
    def __add__(self, o): return self.val + o
    def __radd__(self, o): return o + self.val
    def __rsub__(self, o): return o - self.val
    def __mul__(self, o): return self.val * o
    def __rmul__(self, o): return o * self.val
    def __truediv__(self, o): return self.val / o
    def __rtruediv__(self, o): return o / self.val

class SelfProxy:
    """Intercepts self[...] calls, enforces orthogonal lookbacks, and routes carries."""
    def __init__(self, carries, scan_names):
        self.carries = carries
        self.scan_names = scan_names  # Ordered outer-to-inner

    def __getitem__(self, indices):
        if not isinstance(indices, tuple):
            indices = (indices,)

        current_scan_vals = {}
        lookbacks = []

        for idx in indices:
            if isinstance(idx, IndexProxy):
                current_scan_vals[idx.name] = idx.val
                if idx.offset < 0:
                    if idx.offset == -1:
                        lookbacks.append(idx.name)
                    else:
                        raise ValueError(f"Only '-1' lookbacks are supported, got {idx.offset} for {idx.name}")

        # --- THE STRICT BAN ON DIAGONALS ---
        if len(lookbacks) > 1:
            raise ValueError(
                f"Diagonal lookbacks (e.g., {lookbacks}) are strictly forbidden. "
                "The @vfor framework only supports orthogonal lookbacks "
                "(e.g., self[i-1, j] OR self[i, j-1]) to guarantee safe boundary logic."
            )
        if len(lookbacks) == 0:
            raise ValueError("Must specify exactly one lookback (e.g., i-1).")

        target_scan = lookbacks[0]
        carry = self.carries[target_scan]

        # Route the memory: Slice the outer carry using the inner loop indices
        target_idx = self.scan_names.index(target_scan)
        inner_scans = self.scan_names[target_idx + 1:]

        slice_idx = []
        for inner in inner_scans:
            if inner not in current_scan_vals:
                raise ValueError(f"Missing inner scan index '{inner}' in self[...] lookup.")
            # We strictly use the current loop value (no offsets applied to inner dimensions)
            slice_idx.append(current_scan_vals[inner])

        if slice_idx:
            return carry[tuple(slice_idx)]
        return carry

def vfor(**axes):
    def decorator(f):
        axis_names = list(axes.keys())
        scan_names = [k for k, v in axes.items() if isinstance(v, scan)]
        scan_sizes = [axes[k].size for k in scan_names]
        spatial_names = [k for k, v in axes.items() if isinstance(v, int)]
        spatial_sizes = [axes[k] for k in spatial_names]

        def execute():
            def core_fn(*spatial_args):
                current_args = {name: val for name, val in zip(spatial_names, spatial_args)}
                carries = {}

                def build_scan(depth):
                    if depth == len(scan_names):
                        args_list = [current_args[n] for n in axis_names]
                        return f(*args_list, SelfProxy(carries.copy(), scan_names))

                    name = scan_names[depth]
                    size = scan_sizes[depth]

                    # Trace 1: Initialization
                    current_args[name] = None
                    carries[name] = None
                    init_carry = build_scan(depth + 1)

                    # Trace 2: Main Body
                    def step_fn(carry, val):
                        carries[name] = carry
                        current_args[name] = IndexProxy(name, val, offset=0)
                        next_val = build_scan(depth + 1)
                        return next_val, next_val

                    _, full_grid = jax.lax.scan(step_fn, init_carry, jnp.arange(size))
                    return full_grid

                return build_scan(0)

            # Apply Spatial Vmaps
            mapped_fn = core_fn
            for idx in reversed(range(len(spatial_sizes))):
                in_axes = [None] * len(spatial_sizes)
                in_axes[idx] = 0
                mapped_fn = jax.vmap(mapped_fn, in_axes=tuple(in_axes))

            spatial_index_arrays = [jnp.arange(size) for size in spatial_sizes]
            result = mapped_fn(*spatial_index_arrays)

            # Transpose to user signature
            computed_axes = spatial_names + scan_names
            if computed_axes != axis_names:
                permutation = [computed_axes.index(name) for name in axis_names]
                result = jnp.transpose(result, axes=permutation)

            return result
        return execute()
    return decorator


In [39]:
@vfor(i=scan(4), j=scan(4))
def C(i, j, self):
    if i is None:
        return 1
    elif j is None:
        return 1
    else:
        return self[i-1, j] + self[i, j-1]

C

Array([[ 2,  3,  4,  5],
       [ 3,  6, 10, 15],
       [ 4, 10, 20, 35],
       [ 5, 15, 35, 70]], dtype=int32, weak_type=True)

In [52]:
import numpy as onp

A = jnp.array(onp.random.randn(5,5,10))
B = jnp.array(onp.random.randn(5,20))

@vfor(i=10, j=20)
def C(i, j, self):
    return jnp.linalg.solve(A[:,:,i], B[:,j])

expected = jnp.linalg.solve(A[:,:,3], B[:,4])
onp.allclose(C[3,4,:], expected)

True

In [41]:
@vfor(i=scan(5), j=scan(5))
def C(i, j, self):
    if i is None or j is None:
        return 1
    else:
        return self[i-1, j] + self[i, j-1]

print("Shape:", C.shape)


Shape: (5, 5)


In [36]:
C

Array([[  2,   3,   4,   5,   6],
       [  3,   6,  10,  15,  21],
       [  4,  10,  20,  35,  56],
       [  5,  15,  35,  70, 126],
       [  6,  21,  56, 126, 252]], dtype=int32, weak_type=True)

Thinking out loud, suppose a user writes:

```python
def fun(i, j, k, prev_j, prev_k):
    ...
```

Where `fun` can be anything. Imagine that if you do

```python
vfor(i=5, j=scan(7), k=scan(10))(fun)
```

Then the result should be equivalent to

```python
C = empty((5,7,10))
for i in range(5):
    for j in range(7):
        for k in range(10):
            if j == 0:
                prev_j = fun(i,None,k)
            else:
                prev_j = C[i,j-1,k]
            
            if k==0:
                prev_k = fun(i,j,None)
            else:
                prev_k = C[i,j,k-1]

            carry_k = f3(i, j, k, prev_j, prev_k)
```


If we have no recursion, then

```python
@fvor(i=10, j=20)
def C(i, j, self):
    # any access to self raises an exception
    return fun(i, j)
```

Should just be equivalent to
```python
C = empty((10, 20))
for i in range(10):
    for j in range(20):
        C[i,j] = fun(i,j)
```

If we have a single recursion, then:

```python
@fvor(i=10, j=scan(20))
def C(i, j, self):
    if j is None:
        return fun1(i)
    return fun2(i, j, self[i,j-1])
```

Should just be equivalent to
```python
C = empty((10, 20))
for i in range(10):
    for j in range(20):
        if j==0:
            prev_j = fun1(i)
        else:
            prev_j = C[i,j-1]
        
        C[i,j] = fun2(i, j, prev_i, prev_j)
```

If we have two recursions, then:

```python
@fvor(i=scan(10), j=scan(20))
def C(i, j, C):
    if i is None and j is None:
        # no access to self allowed
        return a
    
    if i is None:
        # access to self[i, j-1] allowed, all other access prohibited
        return fun1(j, C[i, j-1])

    if j is None:
        # access to self[i-1, j] allowed, all other access prohibited
        return fun2(i, C[i-1, j])

    # access to self[i-1, j] and self[i, j-1] allowed, all other access prohibited
    return fun3(i, j, C[i-1,j], C[i,j-1])
```

Should just be equivalent to

```python
# imagine C[i,j] runs from i=-1 to i=9 and j=-1 to j=19
for i in range(-1,10):
    for j in range(-1,20):
        if i==-1 and j==-1:
            C[i,j] = a
        elif i==-1:
            C[i,j] = fun1(j, C[i, j-1])
        elif j==-1:
            C[i,j] = fun2(i, C[i-1, j])
        else:
            C[i,j] = fun3(i, j, C[i-1, j], C[i,j-1])
# truncate C to run from i=0 to i=9 and j=0 to j=19
```



In [61]:
# Test it!
@vfor_ref(i=scan(4), j=scan(4))
def C(i, j, self):
    if i is None and j is None:
        return 0
    if i is None:
        return self[i, j-1] + 1
    if j is None:
        return self[i-1, j] + 5
    return self[i-1, j] + self[i, j-1]

print(C)

[[6 8 11 15]
 [16 24 35 50]
 [31 55 90 140]
 [51 106 196 336]]


In [73]:
import jax.numpy as jnp
import itertools

class scan:
    """Marker class to define a temporal/scanned axis."""
    def __init__(self, size): 
        self.size = size

class ReferenceGrid:
    """
    Manages the padded memory, boundary rules, and JAX export 
    for the reference implementation.
    """
    def __init__(self, axes):
        self.axes = list(axes.values())
        # Offset is 1 for scanned axes (padding), 0 for spatial axes (no padding)
        self.offsets = [1 if isinstance(v, scan) else 0 for v in self.axes]

        # Explicit loop boundaries: -1 to N-1 for scans, 0 to N-1 for spatial
        self.ranges = [
            range(-1, v.size) if offset == 1 else range(0, v) 
            for v, offset in zip(self.axes, self.offsets)
        ]

        # Build the padded recursive list
        padded_shape = [
            v.size + 1 if offset == 1 else v 
            for v, offset in zip(self.axes, self.offsets)
        ]
        self.data_list = self._build_list(padded_shape)

    def _build_list(self, shape):
        if len(shape) == 1:
            return [None] * shape[0]
        return [self._build_list(shape[1:]) for _ in range(shape[0])]

    def map_to_none(self, indices):
        """Replaces the -1 boundary index with None for the user's function."""
        return tuple(
            None if idx == -1 and offset == 1 else idx 
            for idx, offset in zip(indices, self.offsets)
        )

    def __getitem__(self, indices):
        """
        Reads from the padded grid. 
        Shifts mathematical indices (like -1) by the padding offset.
        """
        if not isinstance(indices, tuple): 
            indices = (indices,)

        curr = self.data_list
        for idx, offset in zip(indices, self.offsets):
            if idx is None: 
                idx = -1
            curr = curr[idx + offset]
        return curr

    def __setitem__(self, indices, val):
        """Writes to the padded grid."""
        curr = self.data_list
        for idx, offset in zip(indices[:-1], self.offsets[:-1]):
            curr = curr[idx + offset]
        curr[indices[-1] + self.offsets[-1]] = val

    def to_jax_array(self):
        """Recursively slices off the +1 padding and exports to a JAX array."""
        def slice_padding(lst, depth=0):
            if depth == len(self.offsets):
                return lst
            # If offset is 1 (scanned), start slice at 1 to drop the padding
            start = self.offsets[depth]
            return [slice_padding(item, depth + 1) for item in lst[start:]]

        return jnp.array(slice_padding(self.data_list))


def vfor_ref(**axes):
    """
    A pure Python/NumPy reference implementation of @vfor.
    Uses plain-old loops to execute the grid math exactly as a user 
    would conceptually understand it.
    """
    def decorator(f):
        grid = ReferenceGrid(axes)

        # 1. Run the explicit N-dimensional nested loops
        for indices in itertools.product(*grid.ranges):

            mapped_indices = tuple(None if i == -1 else i for i in indices)
            
            grid[indices] = f(*mapped_indices, grid)

        # 4. Export the final strictly-shaped array
        return grid.to_jax_array()

    return decorator


[[  6   8  11  15]
 [ 16  24  35  50]
 [ 31  55  90 140]
 [ 51 106 196 336]]


In [103]:
import jax
import jax.numpy as jnp

class scan:
    """Marker class to define a temporal/scanned axis."""
    def __init__(self, size): 
        self.size = size

class IndexProxy:
    """
    Wraps dynamic JAX loop variables. 
    Intercepts `-1` offsets for self indexing, but forwards all other math to JAX.
    """
    def __init__(self, name, val, offset=0):
        self.name = name
        self.val = val
        self.offset = offset

    def __sub__(self, other):
        if isinstance(other, int):
            return IndexProxy(self.name, self.val - other, self.offset - other)
        return self.val - other

    # Forward standard mathematical operations to the underlying JAX tracer
    def __add__(self, o): return self.val + o
    def __radd__(self, o): return o + self.val
    def __rsub__(self, o): return o - self.val
    def __mul__(self, o): return self.val * o
    def __rmul__(self, o): return o * self.val
    def __truediv__(self, o): return self.val / o
    def __rtruediv__(self, o): return o / self.val

class SelfProxy:
    """
    Routes `self[...]` calls to the correct JAX memory carries.
    Translates mathematical indices directly into padded JAX array indices.
    """
    def __init__(self, carries, axis_names, scan_names):
        self.carries = carries
        self.axis_names = axis_names
        self.scan_names = scan_names

    def __getitem__(self, indices):
        if not isinstance(indices, tuple): 
            indices = (indices,)

        # Map the user's provided indices to the axis names
        user_idx_map = dict(zip(self.axis_names, indices))

        # 1. Find the target outer loop
        lookbacks = []
        for name in self.scan_names:
            idx = user_idx_map[name]
            if isinstance(idx, IndexProxy) and idx.offset == -1:
                lookbacks.append(name)

        if len(lookbacks) > 1:
            raise ValueError(f"Diagonal lookbacks {lookbacks} are strictly forbidden.")
        if len(lookbacks) == 0:
            raise ValueError("Must specify exactly one lookback (e.g., j-1).")

        target_scan = lookbacks[0]
        target_depth = self.scan_names.index(target_scan)

        # This carry is a PADDED JAX array representing the entire previous sub-grid
        carry = self.carries[target_scan]

        # 2. Slice the padded carry using the inner dimensions
        inner_scans = self.scan_names[target_depth + 1:]
        slice_idx = []
        for inner_name in inner_scans:
            idx = user_idx_map[inner_name]
            # --- THE MAGIC OFFSET MAPPING ---
            if idx is None:
                slice_idx.append(0)           # -1 boundary maps to the 0 index of the padded array
            elif isinstance(idx, IndexProxy):
                slice_idx.append(idx.val + 1) # Main grid maps to Tracer + 1
            else:
                raise ValueError(f"Unexpected index type for {inner_name}")

        if slice_idx:
            return carry[tuple(slice_idx)]
        return carry

def vfor(**axes):
    """
    Compiles tensor comprehensions into Padded JAX vmap/scan graphs.
    """
    def decorator(f):
        axis_names = list(axes.keys())
        scan_names = [k for k, v in axes.items() if isinstance(v, scan)]
        scan_sizes = [axes[k].size for k in scan_names]
        spatial_names = [k for k, v in axes.items() if isinstance(v, int)]
        spatial_sizes = [axes[k] for k in spatial_names]

        def execute():
            def core_fn(*spatial_args):
                spatial_args_dict = {name: val for name, val in zip(spatial_names, spatial_args)}
                current_args = {}
                carries = {}

                def build_scan(depth):
                    # Base Case: Execute the user's math
                    if depth == len(scan_names):
                        args_list = [current_args.get(n, spatial_args_dict.get(n)) for n in axis_names]
                        val = f(*args_list, SelfProxy(carries.copy(), axis_names, scan_names))

                        # --- THE NEW ERROR CATCHER ---
                        if val is None:
                            missing_bounds = [n for n, arg in zip(axis_names, args_list) if arg is None]
                            if missing_bounds:
                                raise ValueError(
                                    f"You forgot to handle the boundary condition! "
                                    f"Your function returned 'None' when evaluating the padding where {missing_bounds} is None."
                                )
                            raise ValueError("Your function returned None, which is not a valid JAX type.")

                        return val

                    name = scan_names[depth]
                    size = scan_sizes[depth]

                    # Trace 1: The Boundary (name = None)
                    current_args[name] = None
                    val_none = build_scan(depth + 1)

                    # Trace 2: The Main Loop (name = Dynamic Tracer)
                    def step_fn(carry, val):
                        carries[name] = carry
                        current_args[name] = IndexProxy(name, val, offset=0)
                        val_grid_step = build_scan(depth + 1)
                        return val_grid_step, val_grid_step

                    # The initial carry for the scan is the computed boundary!
                    _, val_grid = jax.lax.scan(step_fn, val_none, jnp.arange(size))

                    # --- THE PADDING FIX ---
                    # Concatenate the boundary onto the grid so it can be carried safely
                    val_none_expanded = jnp.expand_dims(val_none, axis=0)
                    return jnp.concatenate([val_none_expanded, val_grid], axis=0)

                return build_scan(0)

            # Apply Spatial Vmaps
            mapped_fn = core_fn
            for idx in reversed(range(len(spatial_sizes))):
                in_axes = [None] * len(spatial_sizes)
                in_axes[idx] = 0
                mapped_fn = jax.vmap(mapped_fn, in_axes=tuple(in_axes))

            # Execute the compiled graph
            spatial_index_arrays = [jnp.arange(size) for size in spatial_sizes]
            result = mapped_fn(*spatial_index_arrays)

            # Slice off the padding! (Exactly like the Reference Implementation)
            slices = []
            for name in spatial_names + scan_names:
                if name in scan_names:
                    slices.append(slice(1, None))
                else:
                    slices.append(slice(None))
            result = result[tuple(slices)]

            # Transpose to user signature
            computed_axes = spatial_names + scan_names
            if computed_axes != axis_names:
                permutation = [computed_axes.index(name) for name in axis_names]
                result = jnp.transpose(result, axes=permutation)

            return result
        return execute()
    return decorator


In [104]:
@vfor_ref(i=scan(4), j=scan(4))
def C(i, j, self):
    if i is None and j is None:
        return 0
    if i is None:
        return self[i, j-1] + 1
    if j is None:
        return self[i-1, j] + 5
    return self[i-1, j] + self[i, j-1]

print(C)

@vfor(i=scan(4), j=scan(4))
def C(i, j, self):
    if i is None and j is None:
        return 0
    if i is None:
        return self[i, j-1] + 1
    if j is None:
        return self[i-1, j] + 5
    return self[i-1, j] + self[i, j-1]

print(C)


[[  6   8  11  15]
 [ 16  24  35  50]
 [ 31  55  90 140]
 [ 51 106 196 336]]
[[  6   8  11  15]
 [ 16  24  35  50]
 [ 31  55  90 140]
 [ 51 106 196 336]]


In [105]:
@vfor(i=5)
def C(i, self):
    return i

C

Array([0, 1, 2, 3, 4], dtype=int32)

In [106]:
@vfor(i=scan(5))
def C(i, self):
    return i+self[i-1]

C

TypeError: unsupported operand type(s) for -: 'NoneType' and 'int'

In [114]:
A = jnp.array([1,2,5,7,-9,4])

@vfor(i=scan(6))
def C(i, self):
    if i is None:
        return 0.
    return self[i-1] + A[i]
    
C

IndexError: only integers, slices (`:`), ellipsis (`...`), newaxis (`None`) and integer or boolean arrays are valid indices. Got <__main__.IndexProxy object at 0x7eff12ad1010>